# EquityAgent: An Autonomous Agentic Workflow for Investment Research

**GitHub repository:** https://github.com/nicole-rowley-msaai/MS_AAI_520_NLP_GenAI_Final (branch `agent`)

MS in Applied AI, University of San Diego. AAI-520: NLP and Generative AI,
final project.

**Team:** Data lead, NLP lead, Agent lead (Nicole Rowley), Evaluation lead.

EquityAgent researches a stock ticker end to end: it plans the research,
calls market, macro, news and SEC-filing tools, runs a five-step prompt
chain over the news, routes each task to a specialist, writes a brief,
grades and revises it, reflects on the run, and stores lessons that change
the next run's plan. Demo tickers: NVDA, JPM, XOM.

| Part | Contents | Assignment heading |
|---|---|---|
| A | Data check, Kaggle news EDA, classifier evaluation | Agent Functions and Capabilities |
| B | One agent run with every step printed | Agent Design and Workflows |
| C | Baseline comparison, memory ablation, limitations | Evaluation and Iteration |

**Models.** Planner, router and news chain: `gpt-6-luna`. Writer and
optimizer: `claude-sonnet-5`. Evaluator: `gpt-6-sol` (a different family
from the writer, so the draft is not graded by the model that wrote it).
Embeddings: `all-MiniLM-L6-v2`.


## Setup

In [ ]:
# --- Colab setup -----------------------------------------------------
import os
import sys

from google.colab import drive, userdata

os.chdir("/content")
drive.mount("/content/drive")

REPO = "/content/drive/MyDrive/equity-agent"
REPO_URL = (
    f"https://{userdata.get('GITHUB_TOKEN')}@github.com/"
    "nicole-rowley-msaai/MS_AAI_520_NLP_GenAI_Final.git"
)
if os.path.isdir(os.path.join(REPO, ".git")):
    !git -C {REPO} pull -q
else:
    !git clone -q --branch agent {REPO_URL} {REPO}
%cd {REPO}
sys.path.insert(0, REPO)

SKIP = "google-adk\|dependency resolver"
!pip install -q -r requirements.txt 2>&1 | grep -v "{SKIP}"

SECRET_KEYS = [
    "OPENAI_API_KEY",
    "ANTHROPIC_API_KEY",
    "FRED_API_KEY",
    "NEWSAPI_KEY",
    "EDGAR_USER_AGENT",
    "KAGGLE_USERNAME",
    "KAGGLE_KEY",
]
loaded = []
for key in SECRET_KEYS:
    try:
        os.environ[key] = userdata.get(key)
        loaded.append(key)
    except Exception:
        print(f"Missing secret: {key}")
print("Loaded:", loaded)


In [ ]:
import logging
import warnings

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", module="huggingface_hub")

from src import display as show  # noqa: E402
from src.config import TICKERS  # noqa: E402
from src.evaluation import (  # noqa: E402
    baseline_brief,
    quick_smoke,
    score_against_findings,
    tool_success_rate,
    unsupported_figures,
)
from src.graph import run_agent  # noqa: E402
from src.memory import store  # noqa: E402
from src.tools import safe_call  # noqa: E402
from src.tools.retrieval import index_filings  # noqa: E402

pd.set_option("display.max_colwidth", 120)


# Part A. Data and NLP

## Agent Functions and Capabilities

The agent's tools and the news prompt chain are built and tested before
the agent uses them. This part checks each data source (yfinance, FRED,
NewsAPI, SEC EDGAR), explores the Kaggle financial-news dataset used to
benchmark the sentiment classifier, and compares the LLM classifier with
a keyword baseline. Every tool call goes through `safe_call`, so a failed
API shows as `ok=False` instead of stopping the notebook.


## A1. Data check: does every tool return data?

Each call goes through `safe_call`, so a failed API shows as `ok=False`
instead of stopping the notebook.

In [ ]:
from src.tools.filings import extract_sections, get_latest_10k
from src.tools.macro import get_macro_snapshot
from src.tools.market import compute_technicals, get_price_history
from src.tools.news import get_news


def status(result):
    """Every ToolResult field except the data itself."""
    return {k: v for k, v in vars(result).items() if k != "data"}


checks = []
spy = safe_call("prices", get_price_history, "SPY")
for ticker in TICKERS:
    prices = safe_call("prices", get_price_history, ticker)
    checks.append({"ticker": ticker, **status(prices)})
    if prices.ok and spy.ok:
        print(ticker, compute_technicals(prices.data, spy.data))

macro = safe_call("fred", get_macro_snapshot)
checks.append({"ticker": "-", **status(macro)})
pd.DataFrame(checks)

In [ ]:
macro.data if macro.ok else macro.error

In [ ]:
news = safe_call("news", get_news, "NVDA", "Nvidia")
print(news.ok, news.meta, news.error)
pd.DataFrame(news.data[:5]) if news.ok else None

**10-K sections.** Three filer quirks surfaced during iteration, and
`src/tools/filings.py` handles each:

- **JPMorgan** embeds its whole annual report in the 10-K file, so "Item 7"
  is only a cross-reference. The extractor falls back to the plain heading
  "Management's discussion and analysis" when the Item 7 span is too short.
- **Exxon** reorganized into ExxonMobil Holdings Corp in mid-2026. The
  ticker now maps to a new SEC registrant with no 10-K, so the loader
  retries the prior entity's CIK from `config.PREDECESSOR_CIKS`.
- **Heavy filers** can push a 10-K off the first page of the SEC
  submissions feed, so the loader follows the paginated files.

The `cik` column below shows which registrant each 10-K came from.

In [ ]:
secs = None
for ticker in TICKERS:
    filing = safe_call("edgar", get_latest_10k, ticker)
    if not filing.ok:
        print(ticker, filing.error)
        continue
    secs = extract_sections(filing.data)
    words = {k: len(v.split()) for k, v in secs.items()}
    print(ticker, "cik", filing.data["cik"], filing.data["filed"], words)

if secs:
    print()
    print(secs["mdna"][:500])

## A2. Kaggle news EDA

The labeled dataset is
[Kaggle: Sentiment Analysis for Financial News](https://www.kaggle.com/datasets/ankurzing/sentiment-analysis-for-financial-news)
(the Financial PhraseBank). The cell below downloads it once with the Kaggle
API; add `KAGGLE_USERNAME` and `KAGGLE_KEY` to Colab Secrets, or drop
`all-data.csv` into `data/kaggle/` by hand.

In [ ]:
# Download the Kaggle dataset once; it stays in Drive afterward.
from src.config import KAGGLE_CSV

DATASET = "ankurzing/sentiment-analysis-for-financial-news"

if not KAGGLE_CSV.exists():
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    !kaggle datasets download -d {DATASET} -p data/kaggle --unzip
print("Kaggle CSV present:", KAGGLE_CSV.exists())

In [ ]:
from src.chains.kaggle import (
    LABELS, clean_kaggle, held_out_sample, load_kaggle,
)

raw = load_kaggle()
print("Rows:", len(raw), "| Columns:", list(raw.columns))
print("Exact duplicate headlines:", raw.duplicated("headline").sum())
print("Missing values:", raw.isna().sum().to_dict())
df = clean_kaggle(raw)
print("Rows after cleaning:", len(df))

In [ ]:
counts = df["sentiment"].value_counts().reindex(LABELS)
print((counts / counts.sum()).round(3))

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(counts.index, counts.values, color="#3b6fb6", width=0.6)
for i, v in enumerate(counts.values):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)
ax.set_title("Label balance: mostly neutral")
ax.set_ylabel("Headlines")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
df["words"] = df["headline"].str.split().str.len()
print(df.groupby("sentiment")["words"].describe().round(1))

fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(df["words"], bins=30, color="#3b6fb6")
ax.set_title("Headline length (words)")
ax.set_xlabel("Words")
ax.set_ylabel("Headlines")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
test = held_out_sample(df, n=300)  # fixed seed, stratified
test.to_csv("data/kaggle/test_sample.csv", index=False)
test["sentiment"].value_counts()

## A3. Classifier evaluation: keyword baseline vs LLM

The LLM classifier is scored on the fixed 300-headline held-out sample with
a sentiment-only prompt. The live news chain uses a longer prompt that also
asks for relevance, but the Kaggle headlines have no target company, so that
instruction would only add noise here. Because LLM output varies between
runs (about 2 points of accuracy at n = 300), the score is the mean of
three runs.

In [ ]:
from src.chains.kaggle import keyword_baseline, score, to_articles
from src.chains.news_chain import CLASSIFY_SENTIMENT_SYSTEM, classify

METRICS = ("accuracy", "macro_f1")
N_RUNS = 3  # LLM output varies run to run; report the mean, not one draw

base = score(test["sentiment"], test["headline"].map(keyword_baseline))
print("Keyword baseline:", {k: base[k] for k in METRICS})

articles = to_articles(test)
runs = []
for r in range(N_RUNS):
    labels = classify(articles, system=CLASSIFY_SENTIMENT_SYSTEM)
    pred = [
        labels.get(i, {}).get("sentiment", "neutral")
        for i in range(len(test))
    ]
    runs.append(score(test["sentiment"], pred))
    print(f"LLM run {r + 1}:", {k: runs[-1][k] for k in METRICS})

mean = {k: round(sum(s[k] for s in runs) / N_RUNS, 3) for k in METRICS}
print("LLM mean of", N_RUNS, "runs:", mean)

llm = runs[-1]  # last run's per-class report and confusion matrix
print(llm["report"])

In [ ]:
cm = llm["confusion"]

fig, ax = plt.subplots(figsize=(4, 3.5))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(3), LABELS)
ax.set_yticks(range(3), LABELS)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
for i in range(3):
    for j in range(3):
        color = "white" if cm[i, j] > cm.max() / 2 else "black"
        ax.text(j, i, cm[i, j], ha="center", va="center", color=color)
ax.set_title("LLM classifier confusion matrix")
plt.show()

# Part B. Agent run

## Agent Design and Workflows

The agent is a LangGraph state graph (`src/graph.py`):

```
ticker -> plan -> route -> specialists -> synthesize -> evaluate --pass--> reflect -> memory
                                                            |                 ^
                                                            +--refine--> optimize
```

| Requirement | Where | Printed below |
|---|---|---|
| Plans research steps | `agents/planner.py` | B2 |
| Uses tools dynamically | `agents/toolkit.py`, `tools/` | B3 |
| Routing | `agents/router.py` | B4 |
| Prompt chaining | `chains/news_chain.py` | B5 |
| Evaluator-optimizer | `agents/evaluator.py` | B7 |
| Self-reflection | `agents/evaluator.py: reflect` | B9 |
| Learns across runs | `memory/store.py` | B10 |

This part runs the agent once on NVDA and prints the output of every step
from the run trace, then runs it a second time to show memory in use.
Memory is cleared first so B2 is a true first run.


## B1. Setup check: every tool returns data


In [ ]:
print("Chunks indexed:", index_filings(TICKERS))
pd.DataFrame({t: quick_smoke(t) for t in TICKERS})

## B2. Research plan

The planner reads memory first. On the first run of a ticker there are no
notes; later runs show the lessons it used.

In [ ]:
# Start from empty memory so the first run below is a true first run.
store.NOTES_PATH.unlink(missing_ok=True)
print("Lessons in memory:", len(store.load_notes()))


In [ ]:
TICKER = "NVDA"
notes_before = store.load_notes()

report, trace, state = run_agent(TICKER)

print("Sector:", state["sector"])
print("Hypothesis:", state["plan"].hypothesis)
show.plan_table(trace)

In [ ]:
show.memory_read(trace)

## B3. Tool calls (dynamic tool use)

In [ ]:
show.tool_log(trace)

## B4. Routing decisions

In [ ]:
show.routing_table(trace)

## B5. News prompt chain: output of each step

The digest's `net_sentiment` is not the model's opinion: the chain counts
the per-article labels from the classify step and applies majority rule
with a 15-point margin (`sentiment_counts` is printed with the summary).
Empty `figures` lists in the extract step are expected for headline-only
items; the extractor only reports numbers that appear in the text.


In [ ]:
show.news_chain_steps(trace)

## B6. Specialist findings

In [ ]:
show.findings_table(trace)

## B7. Evaluator-optimizer loop

The evaluator (a different model family from the writer) scores the draft
1-5 on five criteria and lists critiques; the optimizer rewrites the draft
to address them. Every draft gets at least one revision
(`MIN_ITERATIONS = 1`), then the loop stops once the average reaches 4.0
or after three rounds. Iteration 0 is the first draft; iteration 1 is the
revised one.

In [ ]:
show.evaluation_history(state)

In [ ]:
show.critiques(state)

In [ ]:
show.plot_scores(state)
plt.show()

## B8. Final report

In [ ]:
display(show.report_markdown(report))

## B9. Reflection (self-assessment of output and process)

In [ ]:
show.reflection_view(state)

## B10. Memory: what this run taught the agent

New lessons written to `memory/notes.json`, then a second run on the same
ticker to show the planner reading them.

In [ ]:
show.memory_diff(trace)

In [ ]:
report2, trace2, state2 = run_agent(TICKER)
print("Lessons the planner read on run 2:")
display(show.memory_read(trace2))
print("Run 1 steps:", [s.tool for s in state["plan"].steps])
print("Run 2 steps:", [s.tool for s in state2["plan"].steps])
print("Run 1 final score:", state["history"][-1]["average"],
      "| Run 2 final score:", state2["history"][-1]["average"])

# Part C. Evaluation

## Evaluation and Iteration

Part B showed one evaluator-optimizer loop. This part measures the agent
against alternatives, all scored with the same rubric and the same
findings so the comparison is fair.


## C1. Three comparisons

Three comparisons, all scored with the same rubric and the same findings:

1. **Baseline vs agent:** a single prompt with no tools, no plan, no loop.
2. **Numeric spot-check:** figures in each brief that appear in no finding.
3. **Memory on vs off:** the same ticker with memory disabled.

In [ ]:
rows = []
for t in TICKERS:
    rep, tr, st = run_agent(t)
    base = baseline_brief(t)
    base_score = score_against_findings(base, st["findings"])
    findings = st["findings"]
    rows.append(
        {
            "ticker": t,
            "baseline_avg": base_score["average"],
            "agent_first_draft": st["history"][0]["average"],
            "agent_final": st["history"][-1]["average"],
            "iterations": len(st["history"]),
            "tool_success": tool_success_rate(tr),
            "unsupported_baseline": len(unsupported_figures(base, findings)),
            "unsupported_agent": len(
                unsupported_figures(st["draft"], findings)
            ),
        }
    )
results = pd.DataFrame(rows)
results

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.2))
x = list(range(len(results)))
w = 0.27
series = [
    ("baseline_avg", "baseline", "#b5b5b5", -w),
    ("agent_first_draft", "agent, first draft", "#8fb3e0", 0),
    ("agent_final", "agent, final", "#3b6fb6", w),
]
for col, label, color, offset in series:
    ax.bar([i + offset for i in x], results[col], w, label=label, color=color)
ax.set_xticks(x, results["ticker"])
ax.set_ylim(1, 5)
ax.set_ylabel("Rubric average (1-5)")
ax.set_title("Agent beats the single-prompt baseline on every ticker")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=8)
plt.show()

In [ ]:
_, tr_off, st_off = run_agent(TICKER, use_memory=False)
_, tr_on, st_on = run_agent(TICKER, use_memory=True)
pd.DataFrame([
    {"memory": "off", "plan_steps": len(st_off["plan"].steps),
     "notes_read": len(tr_off.select("memory", "read")[0].output),
     "final_score": st_off["history"][-1]["average"]},
    {"memory": "on", "plan_steps": len(st_on["plan"].steps),
     "notes_read": len(tr_on.select("memory", "read")[0].output),
     "final_score": st_on["history"][-1]["average"]},
])

In [ ]:
show.timing_table(trace)

## C2. Limitations

- Figures come from free data sources (`yfinance` is unofficial; NewsAPI
  caps at 100 articles) and are cached, so a run reflects the cache date.
- The evaluator is an LLM and its scores vary by about 0.2 between runs.
- The agent has no view of future events, and the brief is analysis only,
  not investment advice.

## Summary

- **Baseline vs agent.** The single-prompt baseline has no tools, so its
  figures cannot be checked against findings; the agent's can, and the
  `unsupported_*` columns above count figures that appear in no finding.
- **Evaluator-optimizer.** Every draft gets at least one revision; the
  B7 critiques and the B8 report show what changed.
- **Memory.** The score is similar with memory on and off; the benefit
  shows in the plan (extra filing searches and news cross-checks that
  answer the previous run's lessons), not in the rubric average.
- **Robustness.** Each tool call is wrapped so a failed API degrades the
  brief instead of ending the run, and a model that keeps returning
  truncated JSON is retried on the other provider.
